**Dataset Loading and Inspection**

In [11]:
import pandas as pd

cancer_data = pd.read_csv('drive/MyDrive/Datasets For ML/Breast_Cancer_Wisconsin.csv')

print("Columns:", cancer_data.columns)
print("\nShape:", cancer_data.shape)
print("\nDuplicated:", cancer_data.duplicated().sum())
print("\nNull Values:\n", cancer_data.isnull().sum())
print("\nFirst 2 rows:")
cancer_data.head(2)

Columns: Index(['id', 'diagnosis', 'radius_mean', 'texture_mean', 'perimeter_mean',
       'area_mean', 'smoothness_mean', 'compactness_mean', 'concavity_mean',
       'concave points_mean', 'symmetry_mean', 'fractal_dimension_mean',
       'radius_se', 'texture_se', 'perimeter_se', 'area_se', 'smoothness_se',
       'compactness_se', 'concavity_se', 'concave points_se', 'symmetry_se',
       'fractal_dimension_se', 'radius_worst', 'texture_worst',
       'perimeter_worst', 'area_worst', 'smoothness_worst',
       'compactness_worst', 'concavity_worst', 'concave points_worst',
       'symmetry_worst', 'fractal_dimension_worst', 'Unnamed: 32'],
      dtype='object')

Shape: (569, 33)

Duplicated: 0

Null Values:
 id                           0
diagnosis                    0
radius_mean                  0
texture_mean                 0
perimeter_mean               0
area_mean                    0
smoothness_mean              0
compactness_mean             0
concavity_mean               0

,id,diagnosis,radius_mean,texture_mean,perimeter_mean,area_mean,smoothness_mean,compactness_mean,concavity_mean,concave points_mean,...,texture_worst,perimeter_worst,area_worst,smoothness_worst,compactness_worst,concavity_worst,concave points_worst,symmetry_worst,fractal_dimension_worst,Unnamed: 32
0,842302,M,17.99,10.38,122.8,1001.0,0.11840,0.27760,0.3001,0.14710,...,17.33,184.6,2019.0,0.1622,0.6656,0.7119,0.2654,0.4601,0.11890,NaN
1,842517,M,20.57,17.77,132.9,1326.0,0.08474,0.07864,0.0869,0.07017,...,23.41,158.8,1956.0,0.1238,0.1866,0.2416,0.1860,0.2750,0.08902,NaN


**Data Cleaning**

**Drop Unnecessary Columns**

In [12]:
cancer_data = cancer_data.drop(['id', 'Unnamed: 32'], axis=1)
cancer_data.shape

(569, 31)

**Label Encoding**

The target column diagnosis is categorical and must be encoded:

M (Malignant) → 1

B (Benign) → 0

In [13]:
print("Labels Before Encoding: ", cancer_data['diagnosis'].unique())

cancer_data['diagnosis'] = cancer_data['diagnosis'].map({'M': 1, 'B': 0})

print("\nLabels After Encoding: ", cancer_data['diagnosis'].unique())

Labels Before Encoding:  ['M' 'B']

Labels After Encoding:  [1 0]


**Feature–Label Separation**

In [14]:
X, Y = cancer_data.iloc[:, 1:], cancer_data.iloc[:, 0]

**Feature Scaling**

KNN is a distance-based algorithm. Feature scaling is required to ensure that all features contribute equally to distance calculations.
MinMaxScaler is used to scale features between 0 and 1.

In [15]:
from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler()
X_scaled = scaler.fit_transform(X)

X_scaled = pd.DataFrame(X_scaled, columns=X.columns)
X_scaled.head(2)

,radius_mean,texture_mean,perimeter_mean,area_mean,smoothness_mean,compactness_mean,concavity_mean,concave points_mean,symmetry_mean,fractal_dimension_mean,...,radius_worst,texture_worst,perimeter_worst,area_worst,smoothness_worst,compactness_worst,concavity_worst,concave points_worst,symmetry_worst,fractal_dimension_worst
0,0.521037,0.022658,0.545989,0.363733,0.593753,0.792037,0.703140,0.731113,0.686364,0.605518,...,0.620776,0.141525,0.668310,0.450698,0.601136,0.619292,0.568610,0.912027,0.598462,0.418864
1,0.643144,0.272574,0.615783,0.501591,0.289880,0.181768,0.203608,0.348757,0.379798,0.141323,...,0.606901,0.303571,0.539818,0.435214,0.347553,0.154563,0.192971,0.639175,0.233590,0.222878


**Train–Test Split**

The dataset is split into:

80% training data

20% testing data

In [16]:
from sklearn.model_selection import train_test_split

X_train, X_test, Y_train, Y_test = train_test_split(X_scaled, Y, test_size=0.2, random_state=5)

**Radius-Based KNN with Cross-Validation**

To select optimal hyperparameters and avoid overfitting, GridSearchCV is used with Radius-Based KNN.

In [17]:
from sklearn.model_selection import GridSearchCV
from sklearn.neighbors import RadiusNeighborsClassifier
import numpy as np

param_grid = {
    'radius': [1.0, 1.5, 2.0, 2.5, 3.0],
    'weights': ['distance'],
    'metric': ['euclidean']
}

# Added outlier_label to handle samples with no neighbors within the radius
radius_knn = RadiusNeighborsClassifier(outlier_label=0)

grid = GridSearchCV(
    estimator=radius_knn,
    param_grid=param_grid,
    cv=5,
    scoring='accuracy',
    error_score=np.nan
)

grid.fit(X_train, Y_train)

print("Best Parameters:", grid.best_params_)
print("Best Cross-Validation Accuracy:", grid.best_score_)

Best Parameters: {'metric': 'euclidean', 'radius': 1.0, 'weights': 'distance'}
Best Cross-Validation Accuracy: 0.8901098901098902


**Final Model Evaluation**

The best model obtained from cross-validation is evaluated on the test dataset.

In [18]:
best_model = grid.best_estimator_

test_accuracy = best_model.score(X_test, Y_test)
print("Test Accuracy Using Best Model:", test_accuracy)

Test Accuracy Using Best Model: 0.868421052631579


**Model Predictions**

The trained model is used to predict class labels for test data.

In [19]:
Y_pred = best_model.predict(X_test)
print("Predicted Labels:", Y_pred)

Predicted Labels: [1 0 0 0 0 1 0 0 0 0 0 0 1 0 0 0 0 0 0 0 1 0 0 0 0 0 0 1 0 1 1 1 0 1 0 0 1
 0 0 1 0 0 0 1 0 0 1 1 0 0 0 0 0 0 0 1 1 1 0 0 0 1 0 0 0 0 0 0 0 1 0 1 0 0
 0 0 1 0 1 0 1 0 0 0 0 0 1 0 1 1 1 1 0 0 1 1 0 0 0 0 0 1 0 0 0 1 0 1 0 0 0
 0 0 0]


**Performance Evaluation Metrics**

The model is evaluated using standard classification metrics.

In [20]:
from sklearn import metrics

print("Accuracy:", metrics.accuracy_score(Y_test, Y_pred))
print("Precision:", metrics.precision_score(Y_test, Y_pred))
print("Recall:", metrics.recall_score(Y_test, Y_pred))
print("F1 Score:", metrics.f1_score(Y_test, Y_pred))
print("\nClassification Report:\n", metrics.classification_report(Y_test, Y_pred))

Accuracy: 0.868421052631579
Precision: 1.0
Recall: 0.6875
F1 Score: 0.8148148148148148

Classification Report:
               precision    recall  f1-score   support

           0       0.81      1.00      0.90        66
           1       1.00      0.69      0.81        48

    accuracy                           0.87       114
   macro avg       0.91      0.84      0.86       114
weighted avg       0.89      0.87      0.86       114



**Conclusion**

Radius-based KNN accurately classifies breast cancer data when features are properly scaled. Using distance weighting and cross-validation to select the optimal radius improves model performance and prevents overfitting. The final model achieves high accuracy, precision, recall, and F1-score, showing it can reliably distinguish between malignant and benign cases.
